## LOAD SAPS Shapefile

In [5]:
import pandas as pd
import geopandas as gpd

In [6]:
station_data = gpd.read_file("../data/Police_bounds.shp")

station_data.head()



,COMPNT_NM,CREATE_DT,VERSION,geometry
0,BOTSHABELO,20251205,1.4.2,"POLYGON ((26.77137 -29.21403, 26.7733 -29.2217..."
1,KHUBUSIDRIFT,20251205,1.4.2,"POLYGON ((27.72842 -32.53051, 27.72842 -32.531..."
2,STUTTERHEIM,20251205,1.4.2,"POLYGON ((27.50201 -32.44217, 27.49884 -32.465..."
3,MOTHERWELL,20251205,1.4.2,"POLYGON ((25.61061 -33.81772, 25.60713 -33.822..."
4,KWADWESI,20251205,1.4.2,"POLYGON ((25.45586 -33.83107, 25.4666 -33.8334..."


## Load crime stats excel

In [7]:
crime_data = pd.read_excel("../data/2025-2026_-_3rd_Quarter_WEB.xlsx", sheet_name="RAW Data", header=2)
crime_data.columns = (
    crime_data.columns
    .str.strip()
    .str.replace("\n", " ")
    .str.replace(" ", "_")
)
crime_data.head()


,Crime_Category_National_contribution_placement,Crime_Category_Provincial_contribution_placement,Comp_level,Station_Crime_Category,Station,District,Province,Crime_Category,Code,NaN,...,NaN,NaN,October_2025_to__December_2025,National_contribution_placement,National_count_diff_placement,Provincial_contribution_placement,Provincial_count_diff_placement,Count_direction,Count_offence_group,No
0,Murder Station 595,Eastern Cape Murder Station 109,Station,Aberdeen Murder,Aberdeen,Sarah Baartman District,Eastern Cape,Murder,1,1.0,...,2.0,0.0,2.0,595.0,448.0,109.0,83.0,Stabilized,17 Community reported serious Crime,7085.0
1,Attempted murder Station 864,Eastern Cape Attempted murder Station 128,Station,Aberdeen Attempted murder,Aberdeen,Sarah Baartman District,Eastern Cape,Attempted murder,2,0.0,...,0.0,0.0,0.0,864.0,555.0,128.0,84.0,Stabilized,17 Community reported serious crime,7086.0
2,Culpable homicide Station 830,Eastern Cape Culpable homicide Station 120,Station,Aberdeen Culpable homicide,Aberdeen,Sarah Baartman District,Eastern Cape,Culpable homicide,3,1.0,...,0.0,0.0,1.0,830.0,1141.0,120.0,197.0,Decreased,0,7087.0
3,Robbery with aggravating circumstances Station...,Eastern Cape Robbery with aggravating circumst...,Station,Aberdeen Robbery with aggravating circumstances,Aberdeen,Sarah Baartman District,Eastern Cape,Robbery with aggravating circumstances,4,0.0,...,0.0,0.0,0.0,1141.0,721.0,191.0,135.0,Decreased,17 Community reported serious crime,7088.0
4,Common robbery Station 664,Eastern Cape Common robbery Station 77,Station,Aberdeen Common robbery,Aberdeen,Sarah Baartman District,Eastern Cape,Common robbery,6,0.0,...,0.0,1.0,2.0,664.0,416.0,77.0,67.0,Stabilized,17 Community reported serious crime,7089.0


In [8]:
crime_data["Station"] = crime_data["Station"].astype(str)
crime_data = crime_data[crime_data["Station"].str.contains("[A-Za-z]", na=False)]
crime_data["Station"].unique()
crime_data.head()

count_col = [c for c in crime_data.columns if 'October_2025' in str(c)][0]
crime_data_clean = crime_data[['Station', 'Crime_Category', count_col, 'National_contribution_placement', 'Provincial_contribution_placement', 'Count_direction']].copy()
crime_data_clean.columns = ['Station', 'Crime_Type', 'Crime_Count', 'National_placement', 'Provincial_placement', 'Count_direction']
crime_data_clean['Station'] = crime_data_clean['Station'].str.strip().str.upper()
grouped_crime_data = crime_data_clean.groupby(['Station', 'Crime_Type']).agg({
    'Crime_Count': ['sum', 'mean', 'count']
}).reset_index()
grouped_crime_data.head()

Station                                         Crime_Type Crime_Count  \
                                                                       sum   
0  ABERDEEN                17 Community reported serious Crime        76.0   
1  ABERDEEN                                          Abduction         0.0   
2  ABERDEEN                  All theft not mentioned elsewhere        14.0   
3  ABERDEEN                                              Arson         1.0   
4  ABERDEEN  Assault with the intent to inflict grievous bo...        12.0   

               
   mean count  
0  76.0     1  
1   0.0     1  
2  14.0     1  
3   1.0     1  
4  12.0     1